In [0]:
spark.conf.set(
    "fs.azure.account.key.ecomdatalakegen2.dfs.core.windows.net",
    dbutils.secrets.get(scope="azure", key="storage-key").strip()
)

In [0]:
display(dbutils.fs.ls("abfss://bronze@ecomdatalakegen2.dfs.core.windows.net/historical/"))

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

storage_account = "ecomdatalakegen2"
bronze_base = f"abfss://bronze@{storage_account}.dfs.core.windows.net"
silver_base = f"abfss://silver@{storage_account}.dfs.core.windows.net"
checkpoint_base = f"{silver_base}/checkpoints"

# Customers
customers_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")  
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/customers")
    .load(f"{bronze_base}/historical/customers.json")
    .dropDuplicates(["customer_id"])
    .withColumn("signup_date", to_date("signup_date"))
    .withColumn("processed_at", current_timestamp())
)
customers_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_base}/customers_write") \
    .outputMode("append") \
    .start(f"{silver_base}/customers")

# Products
products_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")  
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/products")
    .load(f"{bronze_base}/historical/products.json")
    .dropDuplicates(["product_id"])
    .withColumn("price", col("price").cast("decimal(10,2)"))
    .withColumn("cost", col("cost").cast("decimal(10,2)"))
)
products_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_base}/products_write") \
    .outputMode("append") \
    .start(f"{silver_base}/products")

# Orders
orders_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")  
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/orders")
    .load(f"{bronze_base}/historical/orders.json")
    .dropDuplicates(["order_id"])
    .withColumn("order_date", to_timestamp("order_date"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("price", col("price").cast("decimal(10,2)"))
)
orders_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_base}/orders_write") \
    .outputMode("append") \
    .start(f"{silver_base}/orders")

# Payments
payments_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")  
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/payments")
    .load(f"{bronze_base}/historical/payments.json")
    .dropDuplicates(["payment_id"])
    .withColumn("payment_date", to_timestamp("payment_date"))
    .withColumn("amount", col("amount").cast("decimal(10,2)"))
)
payments_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_base}/payments_write") \
    .outputMode("append") \
    .start(f"{silver_base}/payments")

# Reviews
reviews_df = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiline", "true")  
    .option("cloudFiles.schemaLocation", f"{checkpoint_base}/reviews")
    .load(f"{bronze_base}/historical/reviews.json")
    .dropDuplicates(["review_id"])
    .withColumn("review_date", to_timestamp("review_date"))
    .withColumn("rating", col("rating").cast("int"))
)
reviews_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", f"{checkpoint_base}/reviews_write") \
    .outputMode("append") \
    .start(f"{silver_base}/reviews")

In [0]:
test_data = '{"test": true}'
dbutils.fs.put(
    "abfss://bronze@ecomdatalakegen2.dfs.core.windows.net/historical/test.json",
    test_data,
    overwrite=True
)
print("Test file written.")

In [0]:
display(
    dbutils.fs.ls("abfss://bronze@ecomdatalakegen2.dfs.core.windows.net/historical/")
)

In [0]:
# Delete checkpoint
dbutils.fs.rm("abfss://silver@ecomdatalakegen2.dfs.core.windows.net/_checkpoints", True)

# Re-run the customers stream cell (from the notebook)

In [0]:
display(dbutils.fs.ls("abfss://bronze@ecomdatalakegen2.dfs.core.windows.net/historical/"))